In [1]:
# Cell 1: Setup
print("🚀 Starting PDF Processing...")
print("="*60)

import sys
sys.path.append('..')  # Add parent directory to path

import pdfplumber
import pandas as pd
import json
from pathlib import Path

print("✅ All imports successful!")

🚀 Starting PDF Processing...
✅ All imports successful!


In [2]:
# Cell 2: Load and Explore PDF
pdf_path = "../data/raw/groundwater3_full.pdf"

print(f"📄 Opening PDF: {pdf_path}\n")

with pdfplumber.open(pdf_path) as pdf:
    print(f"Total pages: {len(pdf.pages)}")
    print(f"First page size: {pdf.pages[0].width} x {pdf.pages[0].height}")
    
print("\n✅ PDF loaded successfully!")

📄 Opening PDF: ../data/raw/groundwater3_full.pdf

Total pages: 90
First page size: 612 x 792

✅ PDF loaded successfully!


In [3]:
# Cell 3: Extract Tables from PDF
print("🔍 Extracting tables from PDF...\n")

all_tables = []

with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages, 1):
        print(f"Processing page {page_num}...")
        
        # Extract tables from the page
        tables = page.extract_tables()
        
        for table in tables:
            if table and len(table) > 1:  # Has data
                all_tables.append(table)
                print(f"  ✅ Found table with {len(table)} rows")

print(f"\n✅ Total tables extracted: {len(all_tables)}")

🔍 Extracting tables from PDF...

Processing page 1...
Processing page 2...
  ✅ Found table with 44 rows
Processing page 3...
Processing page 4...
  ✅ Found table with 28 rows
Processing page 5...
  ✅ Found table with 22 rows
Processing page 6...
  ✅ Found table with 33 rows
Processing page 7...
  ✅ Found table with 39 rows
Processing page 8...
  ✅ Found table with 9 rows
Processing page 9...
  ✅ Found table with 24 rows
Processing page 10...
  ✅ Found table with 33 rows
Processing page 11...
  ✅ Found table with 8 rows
Processing page 12...
  ✅ Found table with 32 rows
Processing page 13...
  ✅ Found table with 27 rows
Processing page 14...
  ✅ Found table with 14 rows
Processing page 15...
  ✅ Found table with 20 rows
Processing page 16...
  ✅ Found table with 30 rows
Processing page 17...
  ✅ Found table with 36 rows
Processing page 18...
  ✅ Found table with 20 rows
Processing page 19...
  ✅ Found table with 41 rows
Processing page 20...
  ✅ Found table with 19 rows
Processing page 

In [4]:
# Cell 4: Convert ALL Tables to DataFrame (Smart Combination)
print("📊 Converting ALL tables to DataFrame...\n")

all_data_rows = []

for i, table in enumerate(all_tables, 1):
    try:
        if len(table) > 1:  # Has header and data
            headers = table[0]
            data_rows = table[1:]
            
            # Skip if headers are None or empty
            if not headers or all(h is None for h in headers):
                print(f"⚠️ Table {i}: Skipped (no headers)")
                continue
            
            # Convert each row to a dictionary
            for row in data_rows:
                if row and any(cell for cell in row):  # Row has data
                    row_dict = {}
                    for j, (header, value) in enumerate(zip(headers, row)):
                        # Create unique column names
                        col_name = header if header else f"Column_{j}"
                        row_dict[col_name] = value
                    
                    all_data_rows.append(row_dict)
            
            print(f"Table {i}: Added {len(data_rows)} rows")
    
    except Exception as e:
        print(f"⚠️ Table {i} error: {e}")

# Create DataFrame from all rows
if all_data_rows:
    df = pd.DataFrame(all_data_rows)
    
    print(f"\n✅ Combined ALL tables!")
    print(f"Total DataFrame shape: {df.shape}")
    print(f"Total rows: {len(df)}")
    print(f"Total columns: {len(df.columns)}")
    
    print("\nColumn names:")
    for i, col in enumerate(df.columns[:10], 1):
        print(f"  {i}. {col}")
    if len(df.columns) > 10:
        print(f"  ... and {len(df.columns) - 10} more columns")
    
    print("\nFirst 3 rows:")
    print(df.head(3))
else:
    print("❌ No valid data extracted!")

📊 Converting ALL tables to DataFrame...

Table 1: Added 43 rows
Table 2: Added 27 rows
Table 3: Added 21 rows
Table 4: Added 32 rows
Table 5: Added 38 rows
Table 6: Added 8 rows
Table 7: Added 23 rows
Table 8: Added 32 rows
Table 9: Added 7 rows
Table 10: Added 31 rows
Table 11: Added 26 rows
Table 12: Added 13 rows
Table 13: Added 19 rows
Table 14: Added 29 rows
Table 15: Added 35 rows
Table 16: Added 19 rows
Table 17: Added 40 rows
Table 18: Added 18 rows
Table 19: Added 38 rows
Table 20: Added 13 rows
Table 21: Added 12 rows
Table 22: Added 13 rows
Table 23: Added 13 rows
Table 24: Added 35 rows
Table 25: Added 25 rows
Table 26: Added 38 rows
Table 27: Added 9 rows
Table 28: Added 37 rows
Table 29: Added 9 rows
Table 30: Added 40 rows
Table 31: Added 43 rows
Table 32: Added 9 rows
Table 33: Added 22 rows
Table 34: Added 6 rows
Table 35: Added 6 rows
Table 36: Added 6 rows
Table 37: Added 7 rows
Table 38: Added 14 rows
Table 39: Added 9 rows
Table 40: Added 41 rows
Table 41: Added 1 

In [5]:
# Cell 5: Clean Data (FIXED)
print("🧹 Cleaning data...\n")

print(f"Before cleaning: {df.shape}")

# Remove completely empty rows
df = df.dropna(how='all')

# Clean column names
df.columns = df.columns.str.strip().str.replace('\n', ' ')

# CRITICAL FIX: Make column names unique (this is your main problem!)
print("\n🔧 Fixing duplicate column names...")
cols = pd.Series(df.columns)
for dup in cols[cols.duplicated()].unique():
    print(f"   Found duplicate: '{dup}'")
    # Add numbers to duplicate columns
    cols[cols[cols == dup].index.values.tolist()] = [dup + '_' + str(i) if i != 0 else dup for i in range(sum(cols == dup))]
df.columns = cols

# Clean all text values
print("\n🧹 Cleaning cell values...")
for col in df.columns:
    try:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace('nan', '')
    except:
        pass

print(f"\n✅ Data cleaned!")
print(f"After cleaning: {df.shape}")
print(f"\nFirst 10 column names:")
for i, col in enumerate(df.columns[:10], 1):
    print(f"  {i}. {col}")

🧹 Cleaning data...

Before cleaning: (1057, 144)

🔧 Fixing duplicate column names...
   Found duplicate: 'Sl. No.'
   Found duplicate: 'Projected demand for Domestic and Industrial uses upto 2025'
   Found duplicate: 'Stage of Ground Water Development (%)'
   Found duplicate: 'Net Annual Ground Water Availability'
   Found duplicate: 'States / Union Territories'
   Found duplicate: 'Data Source'
   Found duplicate: 'Annual Ground Water Draft'

🧹 Cleaning cell values...

✅ Data cleaned!
After cleaning: (1057, 144)

First 10 column names:
  1. Sl. No.
  2. States / Union Territories
  3. Annual Replenishable Ground Water Resource
  4. Column_3
  5. Column_4
  6. Column_5
  7. Column_6
  8. Natural Discharge during non- monsoon season
  9. Net Annual Ground Water Availability
  10. Annual Ground Water Draft


In [6]:
# Cell 6: Save Data (SIMPLEST VERSION)
print("💾 Saving data...\n")

# Just save CSV (skip JSON for now)
Path("../data/processed").mkdir(parents=True, exist_ok=True)
csv_path = "../data/processed/groundwater_data.csv"
df.to_csv(csv_path, index=False)

print(f"✅ Saved CSV: {csv_path}")
print(f"🎉 Done! Extracted {len(df)} records")

💾 Saving data...

✅ Saved CSV: ../data/processed/groundwater_data.csv
🎉 Done! Extracted 1057 records


In [7]:
# Cell 7: Preview extracted data
print("👀 Data Preview\n")
print("="*60)

# Show sample records
print("\nSample Data (first 5 records):")
print(df.head())

print("\n" + "="*60)
print("✅ You can now see your extracted groundwater data!")

👀 Data Preview


Sample Data (first 5 records):
  Sl. No. States / Union Territories  \
0    None                       None   
1    None                       None   
2       1                          2   
3                             States   
4       1             Andhra Pradesh   

  Annual Replenishable Ground Water Resource                       Column_3  \
0                             Monsoon Season                           None   
1                    Recharge\nfrom rainfall  Recharge\nfrom other\nsources   
2                                          3                              4   
3                                                                             
4                                      17.25                           6.29   

                  Column_4                       Column_5 Column_6  \
0       Non-monsoon Season                           None    Total   
1  Recharge\nfrom rainfall  Recharge\nfrom other\nsources     None   
2                        5

In [8]:
# Cell 8: Verify Data Coverage
print("📊 Data Coverage Analysis\n")
print("="*60)

print(f"Total records: {len(df)}")
print(f"Total columns: {len(df.columns)}")

# Show all column names
print(f"\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

# Try to find district/state column
state_col = None
for col in df.columns:
    if 'state' in col.lower() or 'district' in col.lower():
        state_col = col
        break

if state_col:
    unique_values = df[state_col].unique()
    print(f"\n✅ Found {len(unique_values)} unique {state_col}:")
    for i, val in enumerate(unique_values[:30], 1):
        if str(val) != 'nan' and str(val).strip():
            print(f"  {i}. {val}")
    
    if len(unique_values) > 30:
        print(f"  ... and {len(unique_values) - 30} more")
else:
    print("\n⚠️ Could not identify state/district column")
    print("First 10 values from column 2:")
    print(df.iloc[:10, 1].tolist())

📊 Data Coverage Analysis

Total records: 1057
Total columns: 144

Column names:
  1. Sl. No.
  2. States / Union Territories
  3. Annual Replenishable Ground Water Resource
  4. Column_3
  5. Column_4
  6. Column_5
  7. Column_6
  8. Natural Discharge during non- monsoon season
  9. Net Annual Ground Water Availability
  10. Annual Ground Water Draft
  11. Column_10
  12. Column_11
  13. Projected demand for Domestic and Industrial uses upto 2025
  14. Ground Water Availability for future irrigation
  15. Stage of Ground Water Development (%)
  16. Sl. No._1
  17. District
  18. Natural
  19. Net Ground Water Availability
  20. Projected demand for Domestic and Industrial uses upto 2025_1
  21. Net Ground Water Availability for Future Irrigation use
  22. Stage of Ground Water Development (%)_1
  23. Provision for Natural Discharges
  24. Net Annual Ground Water Availability_1
  25. Union Territory
  26. Islands
  27. Region
  28. States / Union Territories_1
  29. Total No. of Assesse